# 04 · Repair and verify (model family B on the clean test set)

**What this step does.** For every corrupted training set, repairs the flagged records and retrains a **different model family** (gradient-boosted trees, XGBoost), then scores it on the **clean test set**.

| Repair | What happens to flagged records |
|---|---|
| none | nothing (the damaged baseline) |
| prune | removed from training |
| relabel | label set to the detector's predicted class |
| reweight | kept, with weight 0.2 |
| review | a simulated expert checks each selected record and restores the true label with accuracy 0.8 / 0.9 / 1.0 |

The automatic repairs (prune, relabel, reweight) use flags from the **global** and **stratum** confident-learning audits. Expert **review** (class-conditional and targeted noise only) is tested on three ways of choosing records inside the stratum: the stratum confident-learning flags, the **ranked** review budget, and the **random** review budget (random at accuracy 1.0 only, as the no-model baseline).

**Why model family B matters.** The detector used logistic regression. Scoring with trees on clean labels means the improvement cannot come from the detector grading its own work. This answers the ICTAI circularity criticism directly.

**Scores** (all on the clean test set): AUC, average precision, calibration error (ECE), and recall at a fixed operating point (top-k = the clean positive rate, set on validation), **overall and inside the stratum** (H2, H4). A **clean** model (no noise) is trained once per seed as the reference.

**Inputs:** splits, `results/noisy/*.csv`, `results/detect/*.csv` · **Outputs:** `results/04_verify.csv`.

Runtime: about 30-60 s per run for Diabetes 130 and CDC (about 1 hour each); South German Credit is fast. Set `QUICK = True` for seeds 0-1.

**Which datasets run:** `li.RUN_ONLY` in `li_lib.py` (now: Taiwan credit only). Results for the other datasets are kept: the CSVs are merged, not overwritten.

In [1]:
# --- setup: run this cell first -------------------------------------------
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path.cwd()))          # li_lib.py sits next to the notebooks
import li_lib as li
pd.set_option("display.width", 160, "display.max_columns", 30)
print("li_lib", li.VERSION, "| project folder:", li.ROOT)
li.versions()

li_lib 1.1.0 | project folder: D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity


,package,version
0,numpy,2.2.6
1,pandas,3.0.6
2,scikit-learn,1.9.1
3,xgboost,3.4.1
4,cleanlab,2.9.0
5,lingam,1.13.0
6,causal-learn,0.1.4.8


In [2]:
QUICK = False
seeds = li.SEEDS[:2] if QUICK else li.SEEDS
rows, t0 = [], time.time()
for name, cfg in li.run_configs().items():
    cfg = li.apply_frozen(cfg)
    tr, va, te = (li.load_split(cfg, p) for p in ("train", "val", "test"))
    enc = li.Encoder(cfg).fit(tr)
    Xtr, Xva, Xte = (enc.transform(d).to_numpy(float) for d in (tr, va, te))
    yte, s_te = te[cfg.label].to_numpy(int), li.stratum_mask(te, cfg)
    y_clean = tr[cfg.label].to_numpy(int)
    rate = y_clean.mean()

    def score(y, w, keep, seed, **tags):
        m = li.fit_model_b(Xtr[keep], y[keep], w[keep], seed)
        thr = li.threshold_top_rate(m.predict_proba(Xva)[:, 1], rate)
        rows.append({"dataset": name, "seed": seed, **tags,
                     **li.evaluate(yte, m.predict_proba(Xte)[:, 1], s_te, thr)})

    for seed in seeds:
        ones = np.ones(len(y_clean))
        score(y_clean, ones, ones.astype(bool), seed, mechanism="clean", dose=0.0, audit="-", repair="clean")
        for mech in li.MECHANISMS:
            for dose in li.DOSES:
                run = f"{name}__{mech}__{dose:.2f}__s{seed}"
                nz = pd.read_csv(li.RESULTS / "noisy" / f"{run}.csv")
                dt = pd.read_csv(li.RESULTS / "detect" / f"{run}.csv")
                yn = nz.y_noisy.to_numpy(int)
                P = np.c_[1 - dt.p1, dt.p1]
                score(yn, ones, ones.astype(bool), seed, mechanism=mech, dose=dose, audit="-", repair="none")
                for audit in ("global", "stratum"):
                    flags = dt[f"flag_{audit}"].to_numpy(bool)
                    for rep in ("prune", "relabel", "reweight"):
                        y2, w2, k2 = li.repair(yn, flags, P, rep)
                        score(y2, w2, k2, seed, mechanism=mech, dose=dose, audit=audit, repair=rep)
                if mech in ("targeted", "class_conditional"):
                    for audit, qs in [("stratum", li.REVIEWER_ACCURACY), ("budget_ranked", li.REVIEWER_ACCURACY),
                                      ("budget_random", (1.0,))]:
                        flags = dt[f"flag_{audit}"].to_numpy(bool)
                        for qa in qs:
                            y2, w2, k2 = li.repair(yn, flags, P, "review", reviewer_accuracy=qa,
                                                   y_true=y_clean, seed=seed)
                            score(y2, w2, k2, seed, mechanism=mech, dose=dose, audit=audit, repair=f"review_q{qa}")
        print(f"{name} seed {seed}: done ({time.time()-t0:.0f}s)")
verify = pd.DataFrame(rows)
verify = li.save_merged(verify, "04_verify.csv", show=0)   # all datasets, for the cells below

taiwancredit seed 0: done (96s)
taiwancredit seed 1: done (200s)
taiwancredit seed 2: done (305s)
taiwancredit seed 3: done (411s)
taiwancredit seed 4: done (515s)
kept 3,690 existing rows for ['cdcdiabetes', 'diabetes130', 'southgerman']
wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\04_verify.csv  (4,920 rows x 15 cols)


## H2 quick check: does targeted noise hurt the stratum more than the overall numbers show?
No repair, model B. **Look for:** under targeted noise, `recall_stratum` falls steeply with dose while `auc` (overall) barely moves.

In [3]:
cols = ["auc", "recall", "auc_stratum", "recall_stratum", "recall_rest"]
nr = verify[verify.repair.isin(["none", "clean"])]
nr.groupby(["dataset", "mechanism", "dose"])[cols].mean().round(3)

auc  recall  auc_stratum  recall_stratum  recall_rest
dataset      mechanism         dose                                                         
cdcdiabetes  class_conditional 0.00  0.801   0.431        0.819           0.504        0.420
                               0.03  0.801   0.430        0.818           0.497        0.420
                               0.05  0.800   0.434        0.818           0.503        0.424
                               0.10  0.801   0.430        0.818           0.498        0.420
                               0.20  0.799   0.432        0.816           0.489        0.424
...                                    ...     ...          ...             ...          ...
taiwancredit uniform           0.05  0.791   0.556        0.772           0.671        0.491
                               0.10  0.787   0.549        0.769           0.667        0.483
                               0.20  0.773   0.536        0.759           0.654        0.469
                               0.30  0.751   0.505        0.748           0.621        0.438
                               0.40  0.681   0.424        0.682           0.510        0.375

[88 rows x 5 columns]

## H4 quick check: how much does repair recover?
**Recovery ratio** = (repaired − damaged) / (clean − damaged), for stratum recall. 1.0 = fully restored; 0 = no help; below 0 = repair made it worse. Only computed where the damage is at least 0.05 (otherwise the ratio divides by almost nothing and is meaningless; shown as blank).

In [4]:
def recovery(df, metric):
    clean = df[df.repair == "clean"].groupby(["dataset", "seed"])[metric].mean().rename("clean")
    dmg = df[df.repair == "none"].set_index(["dataset", "seed", "mechanism", "dose"])[metric].rename("damaged")
    rep = df[~df.repair.isin(["none", "clean"])].set_index(["dataset", "seed", "mechanism", "dose"])
    out = rep.join(dmg).join(clean, on=["dataset", "seed"]).reset_index()
    gap = out.clean - out.damaged
    out["recovery_" + metric] = np.where(gap.abs() >= 0.05, (out[metric] - out.damaged) / gap, np.nan)
    return out
rec = recovery(verify, "recall_stratum")
li.save(rec, "04_recovery_recall_stratum.csv", show=0)
(rec[rec.dose >= 0.10].groupby(["dataset", "mechanism", "audit", "repair"])["recovery_recall_stratum"]
   .mean().unstack("repair").round(2))

wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\04_recovery_recall_stratum.csv  (4,480 rows x 18 cols)


repair                                        prune  relabel  review_q0.8  review_q0.9  review_q1.0  reweight
dataset      mechanism         audit                                                                         
cdcdiabetes  class_conditional budget_random    NaN      NaN          NaN          NaN         0.43       NaN
                               budget_ranked    NaN      NaN         0.97         0.83         1.16       NaN
                               global          0.19     0.16          NaN          NaN          NaN      0.19
                               stratum        -1.27    -1.24         0.29         0.26         0.19     -1.22
             targeted          budget_random    NaN      NaN          NaN          NaN         0.07       NaN
                               budget_ranked    NaN      NaN         0.30         0.35         0.38       NaN
                               global         -0.01     0.01          NaN          NaN          NaN      0.00
                               stratum        -0.22    -0.14         0.09         0.12         0.16     -0.19
             uniform           global         -0.10    -0.10          NaN          NaN          NaN      0.03
                               stratum         0.18     0.22          NaN          NaN          NaN     -0.07
diabetes130  class_conditional budget_random    NaN      NaN          NaN          NaN          NaN       NaN
                               budget_ranked    NaN      NaN          NaN          NaN          NaN       NaN
                               global           NaN      NaN          NaN          NaN          NaN       NaN
                               stratum          NaN      NaN          NaN          NaN          NaN       NaN
             targeted          budget_random    NaN      NaN          NaN          NaN         0.07       NaN
                               budget_ranked    NaN      NaN         0.24         0.28         0.29       NaN
                               global         -0.02     0.06          NaN          NaN          NaN      0.01
                               stratum        -0.52    -0.51        -0.00         0.01         0.01     -0.44
             uniform           global          0.67     0.81          NaN          NaN          NaN      0.57
                               stratum        -1.19    -1.25          NaN          NaN          NaN     -1.28
southgerman  class_conditional budget_random    NaN      NaN          NaN          NaN         0.18       NaN
                               budget_ranked    NaN      NaN        -0.09        -0.18        -0.20       NaN
                               global          0.72     0.74          NaN          NaN          NaN      0.19
                               stratum         0.28     0.43         0.03         0.08         0.11      0.11
             targeted          budget_random    NaN      NaN          NaN          NaN        -0.20       NaN
                               budget_ranked    NaN      NaN         0.23         0.27         0.27       NaN
                               global          0.17    -0.10          NaN          NaN          NaN     -0.20
                               stratum        -0.50     0.00         0.13         0.17         0.10     -0.07
             uniform           global         -0.09    -0.06          NaN          NaN          NaN      0.01
                               stratum        -0.27    -0.12          NaN          NaN          NaN     -0.04
taiwancredit class_conditional budget_random    NaN      NaN          NaN          NaN          NaN       NaN
                               budget_ranked    NaN      NaN          NaN          NaN          NaN       NaN
                               global           NaN      NaN          NaN          NaN          NaN       NaN
                               stratum          NaN      NaN          NaN          NaN          NaN       NaN
             targeted          budget_ra

## What to look for before moving on
- [ ] Clean-model AUC is close to the known ceiling (Diabetes readmission is hard to predict: about 0.64-0.68).
- [ ] H2: the gap between stratum recall and overall AUC under targeted noise.
- [ ] H4: which repair and which audit recover most. Note any repair that makes things worse.